### GSM8K


In [2]:
import pandas as pd

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


# Chargement du dataset
df = pd.read_csv("/kaggle/input/datasets/hammouchelyndafarah/gsm8k-100/gsm8k_test_100.csv")

# Prompts baseline
system_prompt_baseline = """You are a math assistant.
Solve the following math problem step by step.
At the end, you must write your final answer on a new line in this exact format:
Answer = <number>"""

def get_user_prompt_baseline(question):
    return f"""Question: '{question}'"""

# Inférence
results = []

for idx, row in df.iterrows():
    messages = [
        {"role": "system", "content": system_prompt_baseline},
        {"role": "user", "content": get_user_prompt_baseline(row["question"])}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=800,
            do_sample=False,
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    results.append({
        "question": row["question"],
        "ground_truth": row["final_answer"],
        "baseline_response": response.strip()
    })

    print(f"Progression : {idx+1}/100")

# Sauvegarde
df_results = pd.DataFrame(results)
df_results.to_csv("baseline_results.csv", index=False)
print("\nDone ! Aperçu :")
print(df_results.head())

Progression : 1/100
Progression : 2/100
Progression : 3/100
Progression : 4/100
Progression : 5/100
Progression : 6/100
Progression : 7/100
Progression : 8/100
Progression : 9/100
Progression : 10/100
Progression : 11/100
Progression : 12/100
Progression : 13/100
Progression : 14/100
Progression : 15/100
Progression : 16/100
Progression : 17/100
Progression : 18/100
Progression : 19/100
Progression : 20/100
Progression : 21/100
Progression : 22/100
Progression : 23/100
Progression : 24/100
Progression : 25/100
Progression : 26/100
Progression : 27/100
Progression : 28/100
Progression : 29/100
Progression : 30/100
Progression : 31/100
Progression : 32/100
Progression : 33/100
Progression : 34/100
Progression : 35/100
Progression : 36/100
Progression : 37/100
Progression : 38/100
Progression : 39/100
Progression : 40/100
Progression : 41/100
Progression : 42/100
Progression : 43/100
Progression : 44/100
Progression : 45/100
Progression : 46/100
Progression : 47/100
Progression : 48/100
P

In [6]:
df_results.to_csv("/kaggle/working/gsm8k_baseline_100.csv", index=False)


In [3]:
def extract_answer(response):
    for line in response.split("\n"):
        if "Answer =" in line:
            return line.split("=")[-1].strip()
    return None

df_results["predicted_answer"] = df_results["baseline_response"].apply(extract_answer)
df_results["correct"] = df_results["predicted_answer"] == df_results["ground_truth"]

accuracy = df_results["correct"].mean()
print(f"Baseline accuracy : {accuracy:.2%}")

Baseline accuracy : 89.00%


In [7]:
import re

def normalize_answer(answer):
    if answer is None:
        return None
    # Supprimer virgules et espaces, convertir en float puis int si possible
    answer = str(answer).replace(",", "").strip()
    try:
        val = float(answer)
        return int(val) if val == int(val) else val
    except:
        return answer

df_results["predicted_norm"] = df_results["predicted_answer"].apply(normalize_answer)
df_results["truth_norm"] = df_results["ground_truth"].apply(normalize_answer)
df_results["correct_norm"] = df_results["predicted_norm"] == df_results["truth_norm"]

accuracy_norm = df_results["correct_norm"].mean()
print(f"Baseline accuracy (normalisée) : {accuracy_norm:.2%}")

Baseline accuracy (normalisée) : 90.00%


In [4]:
# Regarder les erreurs de la baseline
errors = df_results[df_results["correct"] == False]
print(f"Nombre d'erreurs : {len(errors)}")
print(errors[["question", "ground_truth", "predicted_answer"]].to_string())

Nombre d'erreurs : 11
                                                                                                                                                                                                                                                                                                                          question ground_truth predicted_answer
1                                                                                                                                                                                              A team of 4 painters worked on a mansion for 3/8ths of a day every day for 3 weeks. How many hours of work did each painter put in?          189            47.25
2                                                                                                                                                                    It costs $194 per meter to repave a street. Monica's street is 150 meters long. How much more does it cost 

### GSM8K contraintes

In [10]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM



# ============================================================
# Chargement du dataset
# ============================================================

df = pd.read_csv("/kaggle/input/datasets/hammouchelyndafarah/gsm8k-100/gsm8k_test_100.csv")

# ============================================================
# Prompts
# ============================================================

system_prompt_constraints = """You are a constraint extraction assistant.

Given a math question, extract a list of atomic constraints 
that any correct answer must satisfy.

Rules:
- Each constraint must be verifiable independently (true or false)
- Do not solve the question or give the final answer
- Each constraint must be necessary: if violated, the answer is wrong

Output format: a numbered list of constraints only, nothing else."""

def get_user_prompt_constraints(question):
    return f"""Question: '{question}'"""

# ============================================================
# Extraction des contraintes
# ============================================================

results_constraints = []

for idx, row in df.iterrows():
    messages = [
        {"role": "system", "content": system_prompt_constraints},
        {"role": "user", "content": get_user_prompt_constraints(row["question"])}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=256, do_sample=False
        )

    constraints = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    results_constraints.append({
        "question": row["question"],
        "ground_truth": row["final_answer"],
        "constraints": constraints
    })

    print(f"Progression : {idx+1}/100")

# ============================================================
# Sauvegarde
# ============================================================

df_constraints = pd.DataFrame(results_constraints)
df_constraints.to_csv("constraints_results.csv", index=False)
print("Done !")
print(df_constraints.head())

Progression : 1/100
Progression : 2/100
Progression : 3/100
Progression : 4/100
Progression : 5/100
Progression : 6/100
Progression : 7/100
Progression : 8/100
Progression : 9/100
Progression : 10/100
Progression : 11/100
Progression : 12/100
Progression : 13/100
Progression : 14/100
Progression : 15/100
Progression : 16/100
Progression : 17/100
Progression : 18/100
Progression : 19/100
Progression : 20/100
Progression : 21/100
Progression : 22/100
Progression : 23/100
Progression : 24/100
Progression : 25/100
Progression : 26/100
Progression : 27/100
Progression : 28/100
Progression : 29/100
Progression : 30/100
Progression : 31/100
Progression : 32/100
Progression : 33/100
Progression : 34/100
Progression : 35/100
Progression : 36/100
Progression : 37/100
Progression : 38/100
Progression : 39/100
Progression : 40/100
Progression : 41/100
Progression : 42/100
Progression : 43/100
Progression : 44/100
Progression : 45/100
Progression : 46/100
Progression : 47/100
Progression : 48/100
P

### FOLIO



In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Chargement du modèle
model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Réponse : To solve the problem and ensure the answer respects all necessary constraints, let's break down the information provided:

1. **Number of clips sold in April**: Natalia sold clips to 48 friends in April.
2. **Number of clips sold in May**: She sold half as many clips in May as she did in April.

From these points, we can derive the following constraints for the solution:

- Let \( C_A \) be the number of clips sold in April.
- Let \( C_M \) be the number of clips sold in May.
- According to the problem, \( C_A = 48 \).
- Also, \( C_M = \frac{1}{2} \times C_A \).

Now, substituting the value of \( C_A \):

\[ C_M = \frac{1}{2} \times 48 = 24 \]

So, the constraints that the answer must respect are:
- Natalia sold 48 clips in April.
- Natalia sold 24 clips in May.

The total number of clips sold by Natalia over the two months is:

\[ C_A + C_M = 48 + 24 = 72 \]

Therefore, the final answer is that Natalia sold a total of 72 clips.


In [37]:
# Prompt
system_prompt = """You are a rigorous logic assistant. 
Given a set of premises, your task is to determine whether 
a conclusion is True, False, or Unknown.
put your answer between brackets ( ) ."""

user_prompt =  """Premises:
- If people perform in school talent shows often, then they attend and are very engaged with school events.
- People either perform in school talent shows often or are inactive and disinterested members of their community.
- If people chaperone high school dances, then they are not students who attend the school.
- All people who are inactive and disinterested members of their community chaperone high school dances.
- All young children and teenagers who wish to further their academic careers and educational opportunities are students who attend the school.
- Bonnie either both attends and is very engaged with school events and is a student who attends the school, or she neither attends and is very engaged with school events nor is a student who attends the school.

Conclusion: If Bonnie either chaperones high school dances or, if she does not, she performs in school talent shows often, then Bonnie is both a young child or teenager who wishes to further her academic career and educational opportunities and an inactive and disinterested member of the community.

Based solely on these constraints, can the conclusion ever be True?
If it leads to a contradiction in all cases, answer False

 True, False, or Unknown?"""


messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# Inférence
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.1,  # proche de 0 pour réponse déterministe
    )

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:], 
    skip_special_tokens=True
)
print(f"Réponse : {response}")

Réponse : False

Let's break down the premises and see if the conclusion can be true:

1. **Premise 1**: If people perform in school talent shows often, then they attend and are very engaged with school events.
2. **Premise 2**: People either perform in school talent shows often or are inactive and disinterested members of their community.
3. **Premise 3**: If people chaperone high school dances, then they are not students who attend the school.
4. **Premise 4**: All people who are inactive and disinterested members of their community chaperone high school dances.
5. **Premise 5**: All young children and teenagers who wish to further their academic careers and educational opportunities are students who attend the school.
6. **Premise 6**: Bonnie either both attends and is very engaged with school events and is a student who attends the school, or she neither attends and is very engaged with school events nor is a student who attends the school.

**Conclusion**: If Bonnie either chapero

In [35]:
user_prompt_2 = user_prompt_2 = """Premises:
- If people perform in school talent shows often, then they attend and are very engaged with school events.
- People either perform in school talent shows often or are inactive and disinterested members of their community.
- If people chaperone high school dances, then they are not students who attend the school.
- All people who are inactive and disinterested members of their community chaperone high school dances.
- All young children and teenagers who wish to further their academic careers and educational opportunities are students who attend the school.
- Bonnie either both attends and is very engaged with school events and is a student who attends the school, or she neither attends and is very engaged with school events nor is a student who attends the school.

Conclusion: If Bonnie either chaperones high school dances or, if she does not, she performs in school talent shows often, then Bonnie is both a young child or teenager who wishes to further her academic career and educational opportunities and an inactive and disinterested member of the community.

You previously answered: Unknown



Based solely on these constraints, can the conclusion ever be True?
If it leads to a contradiction in all cases, answer False.
Revise your answer: True, False, or Unknown?"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt_2}
]

# Inférence
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=800,
        temperature=0.1,  # proche de 0 pour réponse déterministe
    )

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:], 
    skip_special_tokens=True
)

print(f"Réponse : {response}")

Réponse : False

Let's break down the premises and see if the conclusion can be true:

1. If people perform in school talent shows often, then they attend and are very engaged with school events.
2. People either perform in school talent shows often or are inactive and disinterested members of their community.
3. If people chaperone high school dances, then they are not students who attend the school.
4. All people who are inactive and disinterested members of their community chaperone high school dances.
5. All young children and teenagers who wish to further their academic careers and educational opportunities are students who attend the school.
6. Bonnie either both attends and is very engaged with school events and is a student who attends the school, or she neither attends and is very engaged with school events nor is a student who attends the school.

Now let's analyze the conclusion:
"If Bonnie either chaperones high school dances or, if she does not, she performs in school tale

In [7]:
print(os.listdir("/kaggle/input/datasets/hammouchelyndafarah/foliovalidation-jsonl/"))

['folio-validation.jsonl']


In [2]:
!pip install -q -U bitsandbytes
import importlib
import bitsandbytes
importlib.reload(bitsandbytes)
import json
import re
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Chargement du modèle
model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Chargement d'un seul exemple
examples = []
with open("/kaggle/input/datasets/hammouchelyndafarah/foliovalidation-jsonl/folio-validation.jsonl", "r") as f:
    for line in f:
        examples.append(json.loads(line))

example = examples[0]

# Parser
def parse_response(response):
    match = re.search(r'Answer:\s*(True|False|Unknown|Uncertain)', response, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return "Unparseable"

# Inférence
def get_llm_response(premises, conclusion):
    system_prompt = """You are a rigorous logic assistant. 
Given a set of premises, determine whether a conclusion is 
True, False, or Unknown.
Think step by step, then end your response with exactly:
Answer: True  or  Answer: False  or  Answer: Uncertain"""

    user_prompt = f"""Premises:
{chr(10).join(f"- {p}" for p in premises)}

Conclusion: {conclusion}

Is the conclusion True, False, or Uncertain?"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1000,
            temperature=0.1,
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    return response

# Test sur un seul exemple
raw_response = get_llm_response(example["premises"], example["conclusion"])
parsed = parse_response(raw_response)

# Enregistrement
results = [{
    "index": 0,
    "premises": " | ".join(example["premises"]),
    "conclusion": example["conclusion"],
    "llm_response_raw": raw_response,
    "llm_response_parsed": parsed,
    "correct_label": example["label"]
}]

df = pd.DataFrame(results)
df.to_csv("/kaggle/working/folio_test_one.csv", index=False)

print(df[["index", "conclusion", "llm_response_parsed", "correct_label"]])
print(f"\nRaw response:\n{raw_response}")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

   index                                     conclusion llm_response_parsed  \
0      0  Bonnie performs in school talent shows often.           Uncertain   

  correct_label  
0     Uncertain  

Raw response:
Let's analyze the given premises step by step:

1. **Premise 1**: If people perform in school talent shows often, then they attend and are very engaged with school events.
   - This can be written as: \( P \rightarrow A \) (where \( P \) = performs in school talent shows often, \( A \) = attends and is very engaged with school events).

2. **Premise 2**: People either perform in school talent shows often or are inactive and disinterested members of their community.
   - This can be written as: \( P \lor I \) (where \( I \) = inactive and disinterested members of their community).

3. **Premise 3**: If people chaperone high school dances, then they are not students who attend the school.
   - This can be written as: \( C \rightarrow \neg S \) (where \( C \) = chaperone high school

In [2]:
!pip install -U bitsandbytes>=0.46.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

### Test sur tout le data set

In [7]:
!pip install -q -U bitsandbytes
import importlib
import bitsandbytes
importlib.reload(bitsandbytes)
import json
import re
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Chargement du modèle
model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Chargement du dataset
examples = []
with open("/kaggle/input/datasets/hammouchelyndafarah/foliovalidation-jsonl/folio-validation.jsonl", "r") as f:
    for line in f:
        examples.append(json.loads(line))

# Parser
def parse_response(response):
    match = re.search(r'Answer:\s*(True|False|Uncertain)', response, re.IGNORECASE)
    if match:
        label = match.group(1).capitalize()
        if label.lower() == "Uncertain":
            label = "Uncertain"
        return label
    return "Unparseable"

# Inférence
def get_llm_response(premises, conclusion):
    system_prompt = """You are a rigorous logic assistant. 
Given a set of premises, determine whether a conclusion is 
True, False, or Uncertain.
Think step by step, then end your response with exactly:
Answer: True  or  Answer: False  or  Answer: Uncertain"""

    user_prompt = f"""Premises:
{chr(10).join(f"- {p}" for p in premises)}

Conclusion: {conclusion}

Is the conclusion True, False, or Uncertain?"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1000,
            temperature=0.1,
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    return response

# Boucle sur tous les exemples
results = []
for i, example in enumerate(examples):
    print(f"Example {i+1}/{len(examples)}...")

    raw_response = get_llm_response(example["premises"], example["conclusion"])
    parsed = parse_response(raw_response)

    results.append({
        "index": i,
        "premises": " | ".join(example["premises"]),
        "conclusion": example["conclusion"],
        "llm_response_raw": raw_response,
        "llm_response_parsed": parsed,
        "correct_label": example["label"]
    })
    

    # Sauvegarde tous les 25 exemples
    if (i + 1) % 25 == 0:
        pd.DataFrame(results).to_csv(
            f"/kaggle/working/folio_baseline_checkpoint_{i+1}.csv", 
            index=False
        )
        print(f"✅ Checkpoint sauvegardé à l'exemple {i+1}")

# Sauvegarde finale
df = pd.DataFrame(results)
df.to_csv("/kaggle/working/folio_baseline_final.csv", index=False)

# Accuracy
df["correct"] = df["llm_response_parsed"].str.lower() == df["correct_label"].str.lower()
print(f"\nAccuracy globale : {df['correct'].mean():.2%}")
print(f"\nAccuracy par label :")
print(df.groupby("correct_label")["correct"].mean())
print(f"\nUnparseable : {(df['llm_response_parsed'] == 'Unparseable').sum()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.9 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inco

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Example 1/204...
Example 2/204...
Example 3/204...
Example 4/204...
Example 5/204...
Example 6/204...
Example 7/204...
Example 8/204...
Example 9/204...
Example 10/204...
Example 11/204...
Example 12/204...
Example 13/204...
Example 14/204...
Example 15/204...
Example 16/204...
Example 17/204...
Example 18/204...
Example 19/204...
Example 20/204...
Example 21/204...
Example 22/204...
Example 23/204...
Example 24/204...
Example 25/204...
✅ Checkpoint sauvegardé à l'exemple 25
Example 26/204...
Example 27/204...
Example 28/204...
Example 29/204...
Example 30/204...
Example 31/204...
Example 32/204...
Example 33/204...
Example 34/204...
Example 35/204...
Example 36/204...
Example 37/204...
Example 38/204...
Example 39/204...
Example 40/204...
Example 41/204...
Example 42/204...
Example 43/204...
Example 44/204...
Example 45/204...
Example 46/204...
Example 47/204...
Example 48/204...
Example 49/204...
Example 50/204...
✅ Checkpoint sauvegardé à l'exemple 50
Example 51/204...
Example 52/20

In [10]:
# Sauvegarde finale
df = pd.DataFrame(results)
df.to_csv("/kaggle/working/folio_baseline_final.csv", index=False)


In [13]:
import pandas as pd
import json
# Charger le CSV baseline
df = pd.read_csv("/kaggle/input/datasets/hammouchelyndafarah/folio-baseline-final-csv/folio_baseline_final.csv")

# Charger les exemples originaux
examples = []
with open("/kaggle/input/datasets/hammouchelyndafarah/foliovalidation-jsonl/folio-validation.jsonl", "r") as f:
    for line in f:
        examples.append(json.loads(line))

# Ajouter les premises-FOL
df["premises_fol"] = [" | ".join(example["premises-FOL"]) for example in examples]

# Sauvegarder
df.to_csv("/kaggle/working/folio_baseline_with_fol.csv", index=False)

print(df[["index", "conclusion", "llm_response_parsed", "correct_label", "premises_fol"]].head())

   index                                         conclusion  \
0      0      Bonnie performs in school talent shows often.   
1      1  If Bonnie is either both a young child or teen...   
2      2  If Bonnie either chaperones high school dances...   
3      3                    James has lunch in the company.   
4      4          James does not have lunch in the company.   

  llm_response_parsed correct_label  \
0                True     Uncertain   
1                True          True   
2               False         False   
3               False     Uncertain   
4                True     Uncertain   

                                        premises_fol  
0  ∀x (TalentShows(x) → Engaged(x)) | ∀x (TalentS...  
1  ∀x (TalentShows(x) → Engaged(x)) | ∀x (TalentS...  
2  ∀x (TalentShows(x) → Engaged(x)) | ∀x (TalentS...  
3  ∀x (Meeting(x) → AppearInCompany(x)) | ∀x (Lun...  
4  ∀x (Meeting(x) → AppearInCompany(x)) | ∀x (Lun...  


In [14]:
df

,index,premises,conclusion,llm_response_raw,llm_response_parsed,correct_label,premises_fol
0,0,If people perform in school talent shows often...,Bonnie performs in school talent shows often.,Let's break down the information step by step:...,True,Uncertain,∀x (TalentShows(x) → Engaged(x)) | ∀x (TalentS...
1,1,If people perform in school talent shows often...,If Bonnie is either both a young child or teen...,Let's break down the premises and the conclusi...,True,True,∀x (TalentShows(x) → Engaged(x)) | ∀x (TalentS...
2,2,If people perform in school talent shows often...,If Bonnie either chaperones high school dances...,Let's break down the premises and the conclusi...,False,False,∀x (TalentShows(x) → Engaged(x)) | ∀x (TalentS...
3,3,All employees who schedule a meeting with thei...,James has lunch in the company.,Let's break down the information step by step:...,False,Uncertain,∀x (Meeting(x) → AppearInCompany(x)) | ∀x (Lun...
4,4,All employees who schedule a meeting with thei...,James does not have lunch in the company.,Let's break down the information step by step:...,True,Uncertain,∀x (Meeting(x) → AppearInCompany(x)) | ∀x (Lun...
...,...,...,...,...,...,...,...
199,199,"Ailton Silva, born in 1995, is commonly known ...",No one playing for Nautico is Brazilian.,Let's analyze the premises step by step:\n\n1....,False,False,"BornIn(ailtonsilva, y1995) ∧ CommonlyKnownAs(a..."
200,200,"Ailton Silva, born in 1995, is commonly known ...",Ailton Silva foes not play for a football club.,Let's analyze the premises step by step:\n\n1....,False,False,"BornIn(ailtonsilva, y1995) ∧ CommonlyKnownAs(a..."
201,201,"Ailton Silva, born in 1995, is commonly known ...",Ailton was not loaned out to a football club.,Let's analyze the premises step by step:\n\n1....,False,False,"BornIn(ailtonsilva, y1995) ∧ CommonlyKnownAs(a..."
202,202,"Ailton Silva, born in 1995, is commonly known ...",Ailton Silva played for Fluminense.,Let's analyze the premises step by step:\n\n1....,Uncertain,Uncertain,"BornIn(ailtonsilva, y1995) ∧ CommonlyKnownAs(a..."


### Verification sans contraintes

In [21]:
import pandas as pd
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Charger le dataset
df = pd.read_csv("/kaggle/input/datasets/hammouchelyndafarah/folio-baseline-with-fol-csv/folio_baseline_with_fol.csv")

# Prendre un seul exemple
row = df.iloc[0]

# Parser
def parse_response(response):
    match = re.search(r'Answer:\s*(True|False|Unknown|Uncertain)', response, re.IGNORECASE)
    if match:
        label = match.group(1).capitalize()
        if label.lower() == "unknown":
            label = "Uncertain"
        return label
    return "Unparseable"

# Construire les messages avec les 3 rôles
system_prompt = """You are a rigorous logic assistant. 
Given a set of premises, determine whether a conclusion is 
True, False, or Unknown.
Think step by step, then end your response with exactly:
Answer: True  or  Answer: False  or  Answer: Unknown"""

user_prompt_1 = f"""Premises:
{chr(10).join(f"- {p}" for p in row['premises'].split(' | '))}

Conclusion: {row['conclusion']}

Is the conclusion True, False, or Unknown?"""

user_prompt_2 = f"""Here is how you answered above. Now verify that your answer is correct by checking again

Revise if necessary and answer in the same format:
Answer: True  or  Answer: False  or  Answer: Unknown"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt_1},
    {"role": "assistant", "content": row['llm_response_raw']},  # sa réponse du tour 1
    {"role": "user", "content": user_prompt_2}
]

# Inférence
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        temperature=0.1,
    )

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
).strip()

parsed = parse_response(response)

print(f"Réponse tour 1 : {row['llm_response_parsed']}")
print(f"Réponse tour 2 : {parsed}")
print(f"Correct        : {row['correct_label']}")
print(f"\nRaw tour 2:\n{response}")

Réponse tour 1 : True
Réponse tour 2 : True
Correct        : Uncertain

Raw tour 2:
Let's re-examine the logical steps to ensure the conclusion is correctly derived:

1. **Premise 1**: If people perform in school talent shows often, then they attend and are very engaged with school events.
   - \( P \rightarrow A \)

2. **Premise 2**: People either perform in school talent shows often or are inactive and disinterested members of their community.
   - \( P \lor I \)

3. **Premise 3**: If people chaperone high school dances, then they are not students who attend the school.
   - \( C \rightarrow \neg S \)

4. **Premise 4**: All people who are inactive and disinterested members of their community chaperone high school dances.
   - \( I \rightarrow C \)

5. **Premise 6**: Bonnie either both attends and is very engaged with school events and is a student who attends the school, or she neither attends and is very engaged with school events nor is a student who attends the school.
   - \( (A 

In [22]:
import pandas as pd
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Charger le dataset
df = pd.read_csv("/kaggle/input/datasets/hammouchelyndafarah/folio-baseline-with-fol-csv/folio_baseline_with_fol.csv")



results_tour2 = []

for i, row in df.iterrows():
    print(f"Example {i+1}/{len(df)}...")

    user_prompt_1 = f"""Premises:
{chr(10).join(f"- {p}" for p in row['premises'].split(' | '))}

Conclusion: {row['conclusion']}

Is the conclusion True, False, or Unknown?"""

    user_prompt_2 = f"""Here is how you answered above. Now verify that your answer is correct by checking again

 is your answer correct?
Revise if necessary and answer in the same format:
Answer: True  or  Answer: False  or  Answer: Unknown"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_1},
        {"role": "assistant", "content": row['llm_response_raw']},
        {"role": "user", "content": user_prompt_2}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1000,
            temperature=0.1,
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    results_tour2.append({
        "llm_response_raw_tour2": response,
        "llm_response_parsed_tour2": parse_response(response)
    })

    # Checkpoint tous les 25 exemples
    if (i + 1) % 25 == 0:
        df_temp = df.copy()
        df_temp = df_temp.iloc[:len(results_tour2)]
        df_temp["llm_response_raw_tour2"] = [r["llm_response_raw_tour2"] for r in results_tour2]
        df_temp["llm_response_parsed_tour2"] = [r["llm_response_parsed_tour2"] for r in results_tour2]
        df_temp.to_csv(f"/kaggle/working/folio_tour2_checkpoint_woFO_{i+1}.csv", index=False)
        print(f"✅ Checkpoint sauvegardé à l'exemple {i+1}")

# Ajout des colonnes au dataframe final
df["llm_response_raw_tour2"] = [r["llm_response_raw_tour2"] for r in results_tour2]
df["llm_response_parsed_tour2"] = [r["llm_response_parsed_tour2"] for r in results_tour2]

# Sauvegarde finale
df.to_csv("/kaggle/working/folio_with_tour2_w/o_Fol.csv", index=False)

# Accuracy comparaison
df["correct_tour1"] = df["llm_response_parsed"].str.lower() == df["correct_label"].str.lower()
df["correct_tour2"] = df["llm_response_parsed_tour2"].str.lower() == df["correct_label"].str.lower()

print(f"\nAccuracy Tour 1 : {df['correct_tour1'].mean():.2%}")
print(f"Accuracy Tour 2 : {df['correct_tour2'].mean():.2%}")
print(f"\nAccuracy par label - Tour 1 :")
print(df.groupby("correct_label")["correct_tour1"].mean())
print(f"\nAccuracy par label - Tour 2 :")
print(df.groupby("correct_label")["correct_tour2"].mean())

Example 1/204...
Example 2/204...
Example 3/204...
Example 4/204...
Example 5/204...
Example 6/204...
Example 7/204...
Example 8/204...
Example 9/204...
Example 10/204...
Example 11/204...
Example 12/204...
Example 13/204...
Example 14/204...
Example 15/204...
Example 16/204...
Example 17/204...
Example 18/204...
Example 19/204...
Example 20/204...
Example 21/204...
Example 22/204...
Example 23/204...
Example 24/204...
Example 25/204...
✅ Checkpoint sauvegardé à l'exemple 25
Example 26/204...
Example 27/204...
Example 28/204...
Example 29/204...
Example 30/204...
Example 31/204...
Example 32/204...
Example 33/204...
Example 34/204...
Example 35/204...
Example 36/204...
Example 37/204...
Example 38/204...
Example 39/204...
Example 40/204...
Example 41/204...
Example 42/204...
Example 43/204...
Example 44/204...
Example 45/204...
Example 46/204...
Example 47/204...
Example 48/204...
Example 49/204...
Example 50/204...
✅ Checkpoint sauvegardé à l'exemple 50
Example 51/204...
Example 52/20

OSError: Cannot save file into a non-existent directory: '/kaggle/working/folio_with_tour2_w'

In [23]:
df.to_csv("/kaggle/working/folio_with_tour2_without_Fol.csv", index=False)
